# NLP – Task 1 & Task 2
## IMDb Movie Review Sentiment Analysis and Model Comparison

**Dataset:** IMDb Large Movie Review Dataset (`aclImdb`)

**Task 1:** Data loading, cleaning, text preprocessing, tokenization, and sequence preparation.

**Task 2:** Shared train/test split, text feature extraction, and the implemented SVM baseline.

> **Scope note:** This notebook consolidates the two supplied notebooks in their original workflow order. Task 2 rebuilds its preprocessing and feature representations from the preserved raw review text so that all Task 2 models can share one consistent pipeline. Models not implemented in the supplied Task 2 notebook are not represented as completed.

## Notebook Contents

1. Environment setup and imports
2. **Task 1 –** IMDb data loading and inspection
3. **Task 1 –** Cleaning and text preprocessing
4. **Task 1 –** Tokenization, sequence checks, and padding
5. **Task 2 –** Shared dataset preparation
6. **Task 2 –** Negation-aware preprocessing
7. **Task 2 –** One stratified train/test split
8. **Task 2 –** TF-IDF and Tokenizer/Padding features
9. **Task 2 –** SVM training and predictions

### Important dataset clarification
The supplied notebooks use IMDb labels (`pos`, `neg`). The Task 2 PDF also contains an instruction to merge `Suicidal` into `Depression`, which does not apply to the IMDb labels. The notebook will report whether that label exists rather than silently changing IMDb labels.

## 1. Environment Setup

In [7]:
%pip install pandas numpy matplotlib seaborn scikit-learn nltk tensorflow

Note: you may need to restart the kernel to use updated packages.


### Import Libraries

In [8]:
# Standard library
import os
import re
import time
import random
import string
import warnings
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
)

warnings.filterwarnings("ignore")
print("All libraries imported successfully.")

All libraries imported successfully.


### Set Random Seeds

In [9]:
SEED = 3

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Random seed:", SEED)

Random seed: 3


### Download NLTK Resources

In [10]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources are ready.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20109\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\20109\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20109\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20109\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20109\AppData\Roaming\nltk_data...


NLTK resources are ready.


[nltk_data]   Package omw-1.4 is already up-to-date!


# Task 1 – Data Cleaning

## 2. Load the IMDb Dataset

The dataset folder must contain `train/pos`, `train/neg`, `test/pos`, and `test/neg`.

Expected local structure:

```text
IMDb-Sentiment-Analysis/
├── data/
│   └── aclImdb/
│       ├── train/
│       │   ├── pos/
│       │   └── neg/
│       └── test/
│           ├── pos/
│           └── neg/
└── NLP_Task1_Task2_Combined.ipynb
```

The path resolver below checks common extraction locations, avoiding a machine-specific absolute Windows path.

In [11]:
DATASET_CANDIDATES = [
    Path("data") / "aclImdb"
]
DATASET_PATH = next(
    (
        path for path in DATASET_CANDIDATES
        if all((path / split / label).is_dir()
               for split in ["train", "test"]
               for label in ["pos", "neg"])
    ),
    None
)

if DATASET_PATH is None:
    checked_paths = "\n".join(str(path.resolve()) for path in DATASET_CANDIDATES)
    raise FileNotFoundError(
        "IMDb dataset folders were not found. Extract aclImdb_v1.tar.gz "
        "so that train/pos, train/neg, test/pos, and test/neg exist. "
        f"\nChecked locations:\n{checked_paths}"
    )

print("Dataset folder:", DATASET_PATH.resolve())

Dataset folder: C:\Users\20109\IMDb-Sentiment-Analysis\data\aclImdb


### Load Movie Reviews

In [12]:
def load_reviews(split):
    rows = []

    for sentiment in ["pos", "neg"]:
        folder = DATASET_PATH / split / sentiment

        for file_path in sorted(folder.glob("*.txt")):
            raw_text = file_path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            rows.append({
                "raw_review": raw_text,
                "review": raw_text,
                "sentiment": sentiment
            })

    return rows


data = load_reviews("train") + load_reviews("test")
df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
display(df[["review", "sentiment"]].head())

Dataset shape: (50000, 3)


,review,sentiment
0,Bromwell High is a cartoon comedy. It ran at t...,pos
1,Homelessness (or Houselessness as George Carli...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,pos
3,This is easily the most underrated film inn th...,pos
4,This is not the typical Mel Brooks film. It wa...,pos


### Inspect the Dataset

In [13]:
print("Dataset information:")
print("-" * 50)
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nClass distribution:")
print(df["sentiment"].value_counts())

Dataset information:
--------------------------------------------------
Shape: (50000, 3)

Data types:
raw_review    object
review        object
sentiment     object
dtype: object

Missing values:
raw_review    0
review        0
sentiment     0
dtype: int64

Class distribution:
sentiment
pos    25000
neg    25000
Name: count, dtype: int64


### Check and Remove Missing Values and Duplicates

In [14]:
print("Original shape:", df.shape)
print("Duplicated rows:", df.duplicated(subset=["raw_review", "sentiment"]).sum())

df = df.dropna(subset=["raw_review", "sentiment"]).copy()
df = df.drop_duplicates(
    subset=["raw_review", "sentiment"]
).reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print("Remaining missing values:")
print(df[["raw_review", "sentiment"]].isnull().sum())
print("Remaining duplicates:",
      df.duplicated(subset=["raw_review", "sentiment"]).sum())

Original shape: (50000, 3)
Duplicated rows: 418
Shape after cleaning: (49582, 3)
Remaining missing values:
raw_review    0
sentiment     0
dtype: int64
Remaining duplicates: 0


## 3. Task 1 – Text Preprocessing

The following cells preserve the preprocessing sequence from the original Task 1 notebook. The original raw review text is kept separately in `raw_review` so Task 2 can apply its own consistent preprocessing without losing the source text.

### Convert Text to Lowercase

In [15]:
df["review"] = df["review"].str.lower()
display(df[["review", "sentiment"]].head())

,review,sentiment
0,bromwell high is a cartoon comedy. it ran at t...,pos
1,homelessness (or houselessness as george carli...,pos
2,brilliant over-acting by lesley ann warren. be...,pos
3,this is easily the most underrated film inn th...,pos
4,this is not the typical mel brooks film. it wa...,pos


### Remove Punctuation

In [16]:
df["review"] = df["review"].str.translate(
    str.maketrans("", "", string.punctuation)
)
display(df[["review", "sentiment"]].head())

,review,sentiment
0,bromwell high is a cartoon comedy it ran at th...,pos
1,homelessness or houselessness as george carlin...,pos
2,brilliant overacting by lesley ann warren best...,pos
3,this is easily the most underrated film inn th...,pos
4,this is not the typical mel brooks film it was...,pos


### Remove Extra Spaces

In [17]:
df["review"] = (
    df["review"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
display(df[["review", "sentiment"]].head())

,review,sentiment
0,bromwell high is a cartoon comedy it ran at th...,pos
1,homelessness or houselessness as george carlin...,pos
2,brilliant overacting by lesley ann warren best...,pos
3,this is easily the most underrated film inn th...,pos
4,this is not the typical mel brooks film it was...,pos


### Tokenize Text

In [18]:
df["tokens"] = df["review"].apply(word_tokenize)
display(df[["review", "tokens"]].head())

,review,tokens
0,bromwell high is a cartoon comedy it ran at th...,"[bromwell, high, is, a, cartoon, comedy, it, r..."
1,homelessness or houselessness as george carlin...,"[homelessness, or, houselessness, as, george, ..."
2,brilliant overacting by lesley ann warren best...,"[brilliant, overacting, by, lesley, ann, warre..."
3,this is easily the most underrated film inn th...,"[this, is, easily, the, most, underrated, film..."
4,this is not the typical mel brooks film it was...,"[this, is, not, the, typical, mel, brooks, fil..."


### Remove Stop Words

This cell follows the original Task 1 workflow. The default NLTK English stopword list includes negation terms such as `not` and `no`. Task 2 below uses a revised preprocessing function that explicitly preserves `not`, `no`, and `never`.

In [19]:
stop_words_task1 = set(stopwords.words("english"))

df["tokens"] = df["tokens"].apply(
    lambda tokens: [
        word for word in tokens
        if word not in stop_words_task1
    ]
)

display(df[["review", "tokens"]].head())

,review,tokens
0,bromwell high is a cartoon comedy it ran at th...,"[bromwell, high, cartoon, comedy, ran, time, p..."
1,homelessness or houselessness as george carlin...,"[homelessness, houselessness, george, carlin, ..."
2,brilliant overacting by lesley ann warren best...,"[brilliant, overacting, lesley, ann, warren, b..."
3,this is easily the most underrated film inn th...,"[easily, underrated, film, inn, brooks, cannon..."
4,this is not the typical mel brooks film it was...,"[typical, mel, brooks, film, much, less, slaps..."


### Lemmatize Tokens

In [20]:
lemmatizer_task1 = WordNetLemmatizer()

df["tokens"] = df["tokens"].apply(
    lambda tokens: [
        lemmatizer_task1.lemmatize(word)
        for word in tokens
    ]
)

display(df[["review", "tokens"]].head())

,review,tokens
0,bromwell high is a cartoon comedy it ran at th...,"[bromwell, high, cartoon, comedy, ran, time, p..."
1,homelessness or houselessness as george carlin...,"[homelessness, houselessness, george, carlin, ..."
2,brilliant overacting by lesley ann warren best...,"[brilliant, overacting, lesley, ann, warren, b..."
3,this is easily the most underrated film inn th...,"[easily, underrated, film, inn, brook, cannon,..."
4,this is not the typical mel brooks film it was...,"[typical, mel, brook, film, much, less, slapst..."


## 4. Task 1 – Convert Text to Numerical Sequences

The following cells retain the original Task 1 Keras Tokenizer and Padding workflow. This Task 1 representation is kept separate from Task 2's train-only tokenizer, which is fitted after the train/test split to avoid test-data leakage.

### Convert Text to Numbers (Tokenization)

In [21]:
df["processed_text_task1"] = df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

tokenizer_task1 = Tokenizer()
tokenizer_task1.fit_on_texts(df["processed_text_task1"])

sequences_task1 = tokenizer_task1.texts_to_sequences(
    df["processed_text_task1"]
)

print("Vocabulary size:", len(tokenizer_task1.word_index))
print("First sequence:", sequences_task1[0])

Vocabulary size: 170219
First sequence: [28906, 230, 687, 101, 1869, 6, 1550, 264, 31, 1247, 4259, 48, 3481, 5175, 217, 168, 28906, 230, 1813, 18, 2297, 491, 1247, 19493, 1784, 9422, 5049, 672, 14, 108, 1044, 1247, 21660, 33360, 126, 428, 2867, 264, 546, 672, 117, 176, 672, 3036, 638, 1740, 264, 1079, 12087, 230, 227, 116, 3138, 62, 7698, 4, 1247, 672, 2268, 28906, 230, 416, 36, 643, 423, 30, 28906, 230, 138, 7174, 2113, 118]


### Check Empty Sequences

In [22]:
empty_sequences_task1 = sum(
    1 for sequence in sequences_task1
    if len(sequence) == 0
)

print("Empty sequences:", empty_sequences_task1)

Empty sequences: 0


### Pad Sequences

In [23]:
padded_sequences_task1 = pad_sequences(
    sequences_task1,
    padding="post"
)

print("Padded sequences shape:", padded_sequences_task1.shape)
print("First padded sequence:", padded_sequences_task1[0])

Padded sequences shape: (49582, 1449)
First padded sequence: [28906   230   687 ...     0     0     0]


### Inspect Target Labels

In [24]:
df["sentiment"] = df["sentiment"].astype(str)

print("Sentiment distribution:")
print(df["sentiment"].value_counts())
display(df[["review", "sentiment"]].head())

Sentiment distribution:
sentiment
pos    24884
neg    24698
Name: count, dtype: int64


,review,sentiment
0,bromwell high is a cartoon comedy it ran at th...,pos
1,homelessness or houselessness as george carlin...,pos
2,brilliant overacting by lesley ann warren best...,pos
3,this is easily the most underrated film inn th...,pos
4,this is not the typical mel brooks film it was...,pos


# Task 2 – Text Classification and Model Comparison

## 5. Prepare the Shared Dataset

Task 2 uses the required column names `statement` (input text) and `status` (target label). It starts from the preserved raw reviews, not from the already-cleaned Task 1 text.

In [25]:
df_task2 = df[["raw_review", "sentiment"]].copy()

df_task2 = df_task2.rename(columns={
    "raw_review": "statement",
    "sentiment": "status"
})

print("Task 2 dataset shape:", df_task2.shape)
display(df_task2.head())

Task 2 dataset shape: (49582, 2)


,statement,status
0,Bromwell High is a cartoon comedy. It ran at t...,pos
1,Homelessness (or Houselessness as George Carli...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,pos
3,This is easily the most underrated film inn th...,pos
4,This is not the typical Mel Brooks film. It wa...,pos


### Inspect Required Columns and Class Distribution

In [26]:
required_columns = ["statement", "status"]

missing_columns = [
    column for column in required_columns
    if column not in df_task2.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_task2 = df_task2[required_columns].copy()

print("Shape:", df_task2.shape)
print("\nMissing values:")
print(df_task2.isnull().sum())
print("\nClass distribution:")
print(df_task2["status"].value_counts())

Shape: (49582, 2)

Missing values:
statement    0
status       0
dtype: int64

Class distribution:
status
pos    24884
neg    24698
Name: count, dtype: int64


### Handle Missing Values, Empty Texts, and Duplicates

In [27]:
print("Original shape:", df_task2.shape)

df_task2 = df_task2.dropna(
    subset=["statement", "status"]
).copy()

df_task2["statement"] = df_task2["statement"].astype(str).str.strip()
df_task2["status"] = df_task2["status"].astype(str).str.strip()

df_task2 = df_task2[
    (df_task2["statement"] != "") &
    (df_task2["status"] != "")
].copy()

df_task2 = df_task2.drop_duplicates(
    subset=["statement", "status"]
).reset_index(drop=True)

print("Shape after cleaning:", df_task2.shape)
print("Remaining duplicates:",
      df_task2.duplicated(subset=["statement", "status"]).sum())

Original shape: (49582, 2)
Shape after cleaning: (49582, 2)
Remaining duplicates: 0


### Apply the Task 2 Label-Merging Instruction

The supplied Task 2 PDF mentions merging `Suicidal` into `Depression`. The supplied dataset and both notebooks use IMDb sentiment labels (`pos`, `neg`), so those mental-health labels are not present in this dataset. The conditional code below performs the merge only if the specified source label exists and otherwise leaves IMDb labels unchanged.

In [28]:
df_task2["status"] = df_task2["status"].str.strip().str.lower()

print("Classes before optional merge:")
print(df_task2["status"].value_counts())

if "suicidal" in set(df_task2["status"].unique()):
    df_task2["status"] = df_task2["status"].replace({
        "suicidal": "depression"
    })
    print("\nApplied: suicidal -> depression")
else:
    print(
        "\nNo 'suicidal' label found. "
        "IMDb labels remain unchanged."
    )

print("\nClasses after this step:")
print(df_task2["status"].value_counts())

Classes before optional merge:
status
pos    24884
neg    24698
Name: count, dtype: int64

No 'suicidal' label found. IMDb labels remain unchanged.

Classes after this step:
status
pos    24884
neg    24698
Name: count, dtype: int64


## 6. Task 2 – Negation-Aware Text Preprocessing

Task 2 uses one preprocessing function consistently for every model. Unlike the original Task 1 stopword-removal step, this version preserves `not`, `no`, and `never`.

In [29]:
stop_words_task2 = set(stopwords.words("english"))
negation_words = {"not", "no", "never"}
stop_words_task2 = stop_words_task2 - negation_words

lemmatizer_task2 = WordNetLemmatizer()


def preprocess_text(text):
    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation while preserving apostrophes
    text = re.sub(r"[^\w\s']", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords but preserve negation
    tokens = [
        word for word in tokens
        if word not in stop_words_task2
    ]

    # Lemmatization
    tokens = [
        lemmatizer_task2.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


df_task2["processed_text"] = df_task2["statement"].apply(
    preprocess_text
)

# Remove empty processed texts before splitting
df_task2 = df_task2[
    df_task2["processed_text"].str.strip() != ""
].reset_index(drop=True)

print("Preprocessing completed.")
print("Dataset shape after preprocessing:", df_task2.shape)
display(df_task2[["statement", "processed_text", "status"]].head())

Preprocessing completed.
Dataset shape after preprocessing: (49582, 3)


,statement,processed_text,status
0,Bromwell High is a cartoon comedy. It ran at t...,bromwell high cartoon comedy ran time program ...,pos
1,Homelessness (or Houselessness as George Carli...,homelessness houselessness george carlin state...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,brilliant acting lesley ann warren best dramat...,pos
3,This is easily the most underrated film inn th...,easily underrated film inn brook cannon sure f...,pos
4,This is not the typical Mel Brooks film. It wa...,not typical mel brook film much less slapstick...,pos


### Verify Negation Preservation

In [30]:
test_sentences = [
    "I am not happy",
    "I have no hope",
    "I never feel good"
]

for sentence in test_sentences:
    print("Original:", sentence)
    print("Processed:", preprocess_text(sentence))
    print("-" * 50)

Original: I am not happy
Processed: not happy
--------------------------------------------------
Original: I have no hope
Processed: no hope
--------------------------------------------------
Original: I never feel good
Processed: never feel good
--------------------------------------------------


## 7. Train-Test Split

Split the shared dataset **once** using a stratified 80/20 split and `random_state=3`. All five models in the completed project must use these same train/test rows.

In [31]:
X = df_task2["processed_text"]
y = df_task2["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=3,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training samples: 39665
Testing samples: 9917

Training distribution:
status
pos    19907
neg    19758
Name: count, dtype: int64

Testing distribution:
status
pos    4977
neg    4940
Name: count, dtype: int64


### Encode Target Labels

In [32]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

class_names = label_encoder.classes_
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Encoded training labels:", y_train_encoded[:10])
print("Encoded testing labels:", y_test_encoded[:10])

Classes: ['neg' 'pos']
Number of classes: 2
Encoded training labels: [0 1 1 0 0 0 1 0 0 0]
Encoded testing labels: [1 1 1 0 0 1 1 0 1 0]


## 8. Prepare Text Features

Two feature representations are prepared from the same shared split:

- **TF-IDF** for traditional machine-learning models such as SVM and Naive Bayes.
- **Tokenizer + Padding** for sequence models such as RNN, LSTM, and GRU.

Both vectorizers/tokenizers are fitted on training data only.

### TF-IDF

In [33]:
MAX_FEATURES = 10000

tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF training shape: (39665, 10000)
TF-IDF testing shape: (9917, 10000)
Vocabulary size: 10000


### Tokenizer & Padding

In [34]:
MAX_WORDS = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 64

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Tokenizer vocabulary size:", len(tokenizer.word_index))
print("Training sequence shape:", X_train_pad.shape)
print("Testing sequence shape:", X_test_pad.shape)

Tokenizer vocabulary size: 84098
Training sequence shape: (39665, 200)
Testing sequence shape: (9917, 200)


# Task 2 – Implemented Model

## 9. Support Vector Machine (SVM)

The supplied Task 2 notebook implements and trains a `LinearSVC` model using the shared TF-IDF features.

### Initialize the SVM Model

In [35]:
svm_model = LinearSVC()
print("SVM model initialized successfully.")

SVM model initialized successfully.


### Train the SVM Model and Record Training Time

In [36]:
start_time = time.time()

svm_model.fit(
    X_train_tfidf,
    y_train
)

svm_train_time = time.time() - start_time

print("SVM training completed.")
print(f"Training time: {svm_train_time:.4f} seconds")

SVM training completed.
Training time: 2.6469 seconds


### Generate SVM Predictions

In [37]:
svm_predictions = svm_model.predict(X_test_tfidf)

print("SVM predictions generated.")
print("First 10 predictions:", svm_predictions[:10])
print("First 10 actual labels:", y_test.to_numpy()[:10])

SVM predictions generated.
First 10 predictions: ['pos' 'pos' 'pos' 'neg' 'neg' 'pos' 'pos' 'neg' 'pos' 'pos']
First 10 actual labels: ['pos' 'pos' 'pos' 'neg' 'neg' 'pos' 'pos' 'neg' 'pos' 'neg']


### Compare Actual and Predicted Labels

In [38]:
svm_results = pd.DataFrame({
    "Review": X_test.to_numpy()[:10],
    "Actual Sentiment": y_test.to_numpy()[:10],
    "Predicted Sentiment": svm_predictions[:10]
})

display(svm_results)

,Review,Actual Sentiment,Predicted Sentiment
0,sunday july 16 8 00pm castro san francisco sou...,pos,pos
1,year puzzled julie brown west coast n't househ...,pos,pos
2,went wal mart found film dvd no idea whether m...,pos,pos
3,watched acacia find agree negative review like...,neg,neg
4,shocked 18 page good review one worst movie es...,neg,neg
5,sure 's 50 's drive special n't let fool littl...,pos,pos
6,movie wonderful show man 's work wilderness na...,pos,pos
7,dont ever ever ever consider watching sorry ex...,neg,neg
8,kurt russell elvis plain simple dedication rol...,pos,pos
9,woody allen seems completely devoid inspiratio...,neg,pos
